# GTEx Stage 0 — download, QC, covariate audit

GTEx v10 open-access: gene read counts GCT (stream and subset), sample attributes,
subject phenotypes. Six tissues, RNA-seq analysis freeze only.
Does not modify prior `FINDINGS*.md` or `FALSIFICATION.md`.
Stage 1 is not run from this notebook.

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")
ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(ROOT); sys.path.insert(0, os.path.join(ROOT, "src"))
os.environ["PYTHONUTF8"] = "1"
from gtex_common import GTEX_DIR, GTEX_FIG, GTEX_SEED, RELEASE, stage1_gate
import subprocess, pandas as pd
from pathlib import Path
from IPython.display import Image, display, Markdown
def sh(*args):
    r = subprocess.run([sys.executable, *args], capture_output=True, text=True, encoding="utf-8", errors="replace")
    print(r.stdout)
    if r.returncode:
        print(r.stderr); raise RuntimeError(f"{args[0]} exited with {r.returncode}")
print("ROOT:", ROOT, "| GTEX_SEED:", GTEX_SEED, "| release:", RELEASE)
print("Stage 1 gate:", stage1_gate())

## Frozen tissues and markers

In [ ]:
from gtex_common import TISSUES, NEURONAL_MARKERS, GLIAL_MARKERS, SMAFRZE_RNASEQ
print("freeze:", SMAFRZE_RNASEQ)
print("tissues:")
for t in TISSUES:
    print(" ", t)
print("neuronal:", NEURONAL_MARKERS)
print("glial (frozen a priori):", GLIAL_MARKERS)

## Run Stage 0

In [ ]:
sh("src/gtex_stage0.py")
print((GTEX_DIR / "stage0_report.txt").read_text(encoding="utf-8")[-4000:])

## Cohort and covariate audit

In [ ]:
print((GTEX_DIR / "stage0_cohort.md").read_text(encoding="utf-8")[:6000] if (GTEX_DIR / "stage0_cohort.md").exists() else "no cohort md")
if (GTEX_DIR / "stage0_covariate_audit.csv").exists():
    print(pd.read_csv(GTEX_DIR / "stage0_covariate_audit.csv").to_string(index=False))
if (GTEX_DIR / "stage0_gene_overlap.csv").exists():
    print(pd.read_csv(GTEX_DIR / "stage0_gene_overlap.csv").to_string(index=False))
if (GTEX_DIR / "stage0_composition.csv").exists():
    print(pd.read_csv(GTEX_DIR / "stage0_composition.csv").to_string(index=False))

In [ ]:
for f in ("stage0_age_bins.png", "stage0_rin_vs_age.png", "stage0_isch_vs_age.png", "stage0_composition_vs_age.png"):
    p = GTEX_FIG / f
    if p.exists():
        display(Image(str(p)))

## Manifest and findings

In [ ]:
sh("src/gtex_findings.py")
import json
print(json.dumps(json.loads((GTEX_DIR / "manifest.json").read_text(encoding="utf-8")), indent=1)[:4000])
print(Path("FINDINGS_GTEX.md").read_text(encoding="utf-8")[:5000])